In [1]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import torch
from ultralytics import YOLO

MODEL_DIR = Path("analog_clock/models/clock_bbox_detector_yolo")
TEST_IMAGES_DIR = Path("analog_clock/data/test_images")
OUTPUTS_DIR = MODEL_DIR / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

DATASET_YAML = MODEL_DIR / "dataset.yaml"
PRETRAINED_WEIGHTS = MODEL_DIR / "analog_clock_yolo_model.pt"


In [ ]:
def train_yolo_model(data_yaml: Path = DATASET_YAML, base_model: str = "yolo11n.pt", epochs: int = 50):
    if not data_yaml.exists():
        raise FileNotFoundError(f"Dataset config not found: {data_yaml}")

    model = YOLO(base_model)

    model.train(
        data=str(data_yaml),
        epochs=epochs,
        imgsz=640,
        batch=16,
        device='cuda:0' if torch.cuda.is_available() else 'cpu',
        workers=0,
        amp=True,
        cache=True,
        patience=15,
        project=str(MODEL_DIR / "runs"),
        name='analog_clock_yolo',
        exist_ok=True,
    )
    return model


In [ ]:
# Run only after preparing a proper clock-bbox dataset yaml.
# model = train_yolo_model()


In [ ]:
# Optional: add a fine-tuning cell here after initial training.


In [ ]:
if "model" not in globals():
    if PRETRAINED_WEIGHTS.exists():
        model = YOLO(str(PRETRAINED_WEIGHTS))
    else:
        raise FileNotFoundError(
            f"No loaded model and pretrained weights not found: {PRETRAINED_WEIGHTS}"
        )

image_paths = sorted(
    [p for p in TEST_IMAGES_DIR.iterdir() if p.suffix.lower() in {".png", ".jpg", ".jpeg"}]
)
if not image_paths:
    raise FileNotFoundError(f"No test images found in: {TEST_IMAGES_DIR}")

for image_path in image_paths:
    results = model(str(image_path), verbose=False)
    result = results[0]
    annotated_img = result.plot()

    out_path = OUTPUTS_DIR / f"{image_path.stem}_detected{image_path.suffix}"
    cv2.imwrite(str(out_path), annotated_img)
    print(f"Saved: {out_path}")

    for box in result.boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        class_id = int(box.cls[0])
        label_name = model.names[class_id]
        confidence = float(box.conf[0])
        print(
            f"Found {label_name} at [{x1:.0f}, {y1:.0f}, {x2:.0f}, {y2:.0f}] "
            f"with {confidence:.2f} confidence"
        )


In [ ]:
%matplotlib inline

sample_paths = sorted(OUTPUTS_DIR.glob("*_detected.*"))
if not sample_paths:
    sample_paths = sorted([p for p in TEST_IMAGES_DIR.iterdir() if p.suffix.lower() in {".png", ".jpg", ".jpeg"}])

for image_path in sample_paths:
    image_bgr = cv2.imread(str(image_path))
    if image_bgr is None:
        continue

    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(12, 8))
    plt.imshow(image_rgb)
    plt.axis('off')
    plt.title(image_path.name, fontsize=12)
    plt.show()
    plt.close()


In [ ]:
from pathlib import Path
from ultralytics import YOLO
if "model" not in globals():
    _candidates = [
        Path("analog_clock/models/clock_bbox_detector_yolo/analog_clock_yolo_model.pt"),
        Path("analog_clock/models/clock_bbox_detector_yolo/runs/detect/weights/best.pt"),
    ]
    _found = next((p for p in _candidates if p.exists()), None)
    if _found is not None:
        model = YOLO(str(_found))
        print(f"Loaded model from {_found}")
    else:
        raise FileNotFoundError("No trained model found. Run training first or place weights in expected path.")

print(getattr(model, "ckpt_path", "ckpt_path not available"))


In [ ]:
from pathlib import Path
from ultralytics import YOLO
if "model" not in globals():
    _candidates = [Path(p) for p in [
        "analog_clock/models/clock_bbox_detector_yolo/analog_clock_yolo_model.pt",
        "analog_clock/models/clock_bbox_detector_yolo/runs/detect/weights/best.pt",
    ]]
    _found = next((p for p in _candidates if p.exists()), None)
    if _found is not None:
        model = YOLO(str(_found))
        print(f"Loaded model from {_found}")
    else:
        raise FileNotFoundError("No trained model found. Run training first or place weights in expected path.")
torch.save(model.model.state_dict(), str(MODEL_DIR / "clock_bbox_detector_state_dict.pt"))
